# 08b · 图神经网络：让每个残基看看邻居

## 这一章在整个项目里的位置

```
00–07 ✅ → [08 深度学习] → 09 全蛋白组预测 → 10 对外 API
              │
              ├─ 08a PyTorch 入门：ESM 向量 + 小网络 ✅
              ├─ 08b 图神经网络：AlphaFold 结构图   ← 你在这里
              └─ 08c 微调 ESM-2（LoRA）
```

## 背景：为什么需要这一章

08a 的网络有一个明显的短板：它**一次只看一个残基**。可是蛋白的功能位点往往是三维空间里凑在一起的几个残基，它们在序列上可能隔得很远。一个残基重不重要，要看它挨着谁。

07d 早就把"谁挨着谁"存好了：每个残基在 10 Å 以内有哪些邻居。把蛋白看成一张**图**，在图上做学习的网络叫**图神经网络**（GNN），这一章用它。

**先把结果说在前面：图网络没有让我们的成绩变好。** 但这一章回答了一个更值钱的问题：**什么时候该用图网络，什么时候不该用，为什么。** 能说清楚这个，比多 0.01 的成绩有用。

## 你在哪里操作

```
你：在这个 notebook 里运行一格
   ├─ 训练在你的 Mac 上，这一章用 CPU（原因见第 4 步，是一次真实的事故）
   └─ 成绩记到 Databricks 的 MLflow，和前两章记在同一个本子里
看结果：这里的输出；Databricks 网页的 Experiments
```

## 这一章做的 7 件事

| # | 做什么 | 比喻 |
|---|---|---|
| 1 | 蛋白是一张图 | 一张"谁挨着谁"的关系网 |
| 2 | 图神经网络在做什么 | 每个人先听一圈邻居怎么说，再更新自己的看法 |
| 3 | 把蛋白变成图 | 把关系网录进电脑 |
| 4 | 对照实验：2 × 2 | 一次只改一样东西 |
| 5 | 看结果 | 有时有用，有时帮倒忙 |
| 6 | 为什么：探针实验 | 查一查 ESM 的笔记里已经写了什么 |
| 7 | 图网络能帮上集成吗 | 再请一位大夫会诊，看有没有新意见 |

**花费：0**。**时间**：约 20 分钟，其中训练约 13 分钟。

---

## 准备

运行下面这一格。**不用看代码。**

In [ ]:
import os, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", category=FutureWarning)
import pandas as pd
import torch
from databricks.sdk import WorkspaceClient
from qafi_platform import lakehouse, tracking
from qafi_platform.lakehouse import sql
from qafi_core import baseline, graph, neural
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
w = WorkspaceClient(profile="qafi")
CATALOG = "qafi_dev"
EXPERIMENT = f"/Users/{w.current_user.me().user_name}/qafi-variant-effect"   # 和 07f、08a 是同一个 MLflow 实验
ESM2 = "08e4846e537177426273712802403f7ba8261b6c"
MODELS = ROOT / "data/models"
MODELS.mkdir(parents=True, exist_ok=True)
print("Databricks:", w.config.host)
print("这一章训练用的处理器:", graph.device())

---

## 第 1 步 · 蛋白是一张图

### 在干什么
把"蛋白是一张图"这句话变成具体的数据，并取到本机。

### 背景
**图**（graph）在计算机里就是两样东西：
- **节点**（node）：一个个的东西。这里是**残基**
- **边**（edge）：哪两个东西有关系。这里是**两个残基在三维空间里相距不到 10 Å**

蛋白折叠以后，序列上隔得很远的残基可以在空间里挨在一起。看一个真实的例子，PTEN（一个抑癌基因，在我们的训练集里）的第 124 位，催化位点的半胱氨酸：

### 执行
取三样东西：训练突变（和 08a 一样）、每个残基的邻居和结构特征、ESM 向量表（08a 做好的，直接打开）。

In [ ]:
lakehouse.download(w, f"SELECT * FROM {CATALOG}.gold.fct_variant_features WHERE split = 'train'", MODELS / "train_variants.parquet")
train = pd.read_parquet(MODELS / "train_variants.parquet")
VERSION = sql(w, f"DESCRIBE HISTORY {CATALOG}.gold.fct_variant_features LIMIT 1")["version"][0]
lakehouse.download(w, f"""
    SELECT r.uniprot_id, r.position, r.aa, r.plddt, r.rsa, r.contacts_8, r.contacts_12, r.hse_up, r.hse_down, s.neighbours
    FROM {CATALOG}.gold.dim_residue r
    JOIN {CATALOG}.gold.protein_split p ON p.uniprot_id = r.uniprot_id AND p.split = 'train'
    LEFT JOIN {CATALOG}.silver.residue_structure s ON s.uniprot_id = r.uniprot_id AND s.position = r.position AND s.aa = r.aa""",
    MODELS / "train_residues.parquet")
residues = pd.read_parquet(MODELS / "train_residues.parquet")
parts = ROOT / f"data/esm2-650m/{ESM2}-labelled/embeddings"
if not parts.exists() and not (MODELS / "train_embeddings.npy").exists():   # 换了电脑又没跑过 08a：从平台取（约 5 分钟）
    parts = MODELS / "embeddings.parquet"
    lakehouse.download(w, f"""SELECT e.uniprot_id, e.position, e.embedding FROM {CATALOG}.silver.residue_embedding e
                              JOIN {CATALOG}.gold.protein_split s USING (uniprot_id) WHERE s.split = 'train'""", parts)
table, index = neural.embedding_table(parts, train.uniprot_id.unique(), MODELS / "train_embeddings.npy")
pten = residues[residues.uniprot_id == "P60484"].set_index("position")
print(f"{len(train):,} 个训练突变；{len(residues):,} 个残基，平均每个有 {residues.neighbours.map(len).mean():.1f} 个邻居")
print("PTEN 第 124 位:", pten.loc[124, "aa"], "| rsa", pten.loc[124, "rsa"], "| 10 Å 内的邻居:", sorted(int(n) for n in pten.loc[124, "neighbours"]))

### 结果怎么看
- **31 万**个残基，平均每个有 **11 个**空间邻居
- PTEN 第 124 位是 **C**（半胱氨酸），`rsa` 0.04：埋在蛋白里面。它有 **27 个**邻居
- 邻居里有 122、123、125、126……这些是序列上紧挨着它的，不稀奇
- **但还有 34–37、68–74、88–92**：序列上隔了三五十个位置，折叠以后却凑到了它身边。这些才是"图"带来的新信息，光看序列的前后几位看不到

这一步多取了 6 个结构特征（`plddt`、`rsa`、接触数……）。后面有一个实验要**完全不用 ESM**，只靠这些。

### 用了什么技术，为什么
边的定义用的是 07d 定的：两个残基的 C-beta 原子相距不到 10 Å。这是结构生物信息学里最常用的"接触"定义之一。还有别的定法（比如每个残基只连最近的 k 个），我们用现成的。

### 面试怎么说
> "A protein becomes a graph with residues as nodes and an edge between residues whose C-beta atoms are within 10 angstrom in the AlphaFold model, so residues far apart in sequence but close in space are connected."

---

## 第 2 步 · 图神经网络在做什么

### 在干什么
弄懂图网络的核心动作：**消息传递**（message passing）。只有这一个新概念。

### 背景
比喻：**每个人先听一圈邻居怎么说，再更新自己的看法。**

每个残基手里有一张卡片，上面写着描述自己的一串数。**一层图层**做的事只有一件：

> 每个残基把**邻居们的卡片取个平均**，和**自己原来的卡片**合在一起，写成一张新卡片。

用数字走一遍。三个残基 A、B、C 互相挨着，为了简单，卡片上只有一个数：

| | 原来的卡片 | 邻居 | 邻居的平均 | 新卡片（假设是"自己 + 邻居平均"） |
|---|---|---|---|---|
| A | 1 | B、C | (3 + 5) / 2 = 4 | 1 + 4 = 5 |
| B | 3 | A、C | (1 + 5) / 2 = 3 | 3 + 3 = 6 |
| C | 5 | A、B | (1 + 3) / 2 = 2 | 5 + 2 = 7 |

做完一层，A 的卡片里就有了 B 和 C 的信息。真实的网络里，"合在一起"不是简单相加，而是各乘一组**要训练的权重**再相加，所以网络能自己学会"邻居的哪些信息要紧"。

**叠两层会怎样**：第二层里，A 听到的是邻居们**已经更新过**的卡片，而那些卡片里已经有了"邻居的邻居"的信息。所以：
- 1 层：每个残基知道 10 Å 以内的事
- 2 层：知道两步以内的事（最远约 20 Å）
- 层数 = 消息能传几步

**和 08a 的关系**：把 08a 的网络从中间切开，插进去两层这样的图层，就是这一章的网络。图层数设成 0，就原样退回到 08a。所以两者可以公平地比，差别只有一个：**看不看邻居**。

```
每个残基的输入
      │  编码成 256 个数
      ▼
 [ 图层 × 2 ]          ← 这一章新加的。设成 0 层，就是 08a
      │
      ▼
 拼上突变的特征 → 128 → 20 个输出 → 取"换成的那一种"
```

### 用了什么技术，为什么
- **GraphSAGE**：我们用的图层的名字，是最朴素的一种，做的就是上面说的"自己 + 邻居的平均"。还有更花哨的（让网络自己决定重点听哪个邻居的 GAT，把距离也用上的各种变体）。我在一折上试过把距离也喂进去的那种，没有更好，所以用最简单的
- **PyTorch Geometric**（PyG）：在 PyTorch 上写图神经网络的标准库。它管两件麻烦事：图层的实现，以及怎么把大小不一的图凑成一批（下一步）

### 公司里什么情况怎么做
图神经网络用在"关系"本身就是信息的地方：分子（原子和化学键）、蛋白结构、社交网络、知识图谱、推荐系统（用户和商品）。药物研发里预测小分子性质的模型，大多是图神经网络。

### 面试怎么说
> "A graph layer is one round of message passing: each node combines its own vector with an aggregate of its neighbours' vectors through learned weights. Stacking k layers lets information travel k hops. I used GraphSAGE with mean aggregation in PyTorch Geometric."

---

## 第 3 步 · 把蛋白变成图

### 在干什么
把每个蛋白做成 PyG 认识的图，并且把用不上的部分裁掉。

### 背景
PyG 里一张图是一个 `Data` 对象，最要紧的两样：
- **节点上的数据**：每个节点一行。我们放了三样：ESM 向量、氨基酸种类、6 个结构特征
- **`edge_index`**：所有的边，两行的表。第一行是"谁说"，第二行是"谁听"。比如 `[[0, 1], [1, 0]]` 就是节点 0 和节点 1 互相听

我们还在每张图上挂了它的**突变**：哪个节点、换成什么、标签多少。

### 用了什么技术，为什么
**只留用得上的残基。** 这里有一个省掉六分之五计算量的观察：

- 大多数实验只测了蛋白的一个结构域。29 万个训练残基里，**有标签的只有 3.1 万个**
- 两层图层，消息最多传两步。离有标签的残基**两步以外**的残基，不管它是什么，都影响不到任何一个预测
- 所以把它们裁掉，结果**一模一样**，一个数都不差。有一个测试专门验证这一点：同一个网络，在裁过的图和完整的图上，预测完全相同

**怎么把大小不一的图凑成一批**：08a 每批是 4,096 个突变，整整齐齐。图不行，每个蛋白大小都不一样。PyG 的 `DataLoader` 的办法很巧：把几张图**并排放在一起，当成一张大图**，图与图之间没有边，所以消息不会串门。节点重新编号，边跟着改。这就是 08a 说"以后会用上 `DataLoader`"的地方。

### 执行

In [ ]:
graphs = graph.protein_graphs(train, residues, table, index)
nodes, edges = sum(g.num_nodes for g in graphs.values()), sum(g.edge_index.shape[1] for g in graphs.values())
print(f"{len(graphs)} 张图，共 {nodes:,} 个节点、{edges:,} 条边（裁剪前是 {residues.uniprot_id.isin(set(graphs)).sum():,} 个节点）")
graphs["P60484"]

### 结果怎么看
- **349 张图**，一个训练蛋白一张
- 裁剪以后剩 **49,842 个节点、66 万条边**；裁剪前是 29.3 万个节点。只留了 17%
- 最后打印的是 PTEN 这张图。几项的意思：
  - `embedding=[节点数, 1280]`：每个节点的 ESM 向量
  - `structure=[节点数, 6]`：每个节点的 6 个结构特征
  - `edge_index=[2, 边数]`：所有的边
  - `variant_...`：挂在这张图上的突变。`variant_node` 是它在第几个节点上

### 公司里什么情况怎么做
- **先问"哪些计算根本不会影响结果"**，再谈加速。这次裁剪没有牺牲任何精度，速度却快了好几倍。这类优化要配一个"结果不变"的测试，否则没人敢信
- 图数据最容易出错的地方是**编号**：节点一旦重新编号，所有指向节点的东西（边、突变所在的节点）都要跟着改。我们为此专门写了测试

### 面试怎么说
> "With two message-passing layers a prediction depends only on nodes within two hops, so I keep just the two-hop neighbourhood of labelled residues: 17% of the nodes, identical outputs, verified by a test. Graphs are batched as one disconnected graph with re-indexed nodes."

---

## 第 4 步 · 对照实验：2 × 2

### 在干什么
设计一个实验，干净地回答："看邻居"到底有没有用。

### 背景
比喻：**一次只改一样东西。** 如果直接拿图网络和 08a 的网络比，就算成绩不一样，也说不清是因为看了邻居，还是因为别的什么改动。所以做一张 2 × 2 的表，四个网络，**每一行里只有"看不看邻居"这一样不同**：

| | 不看邻居（0 层图层） | 看邻居（2 层图层） |
|---|---|---|
| **输入有 ESM 向量** | `gnn_esm_0_layers` | `gnn_esm_2_layers` |
| **输入没有 ESM，只有结构** | `gnn_structure_0_layers` | `gnn_structure_2_layers` |

第二行是关键的对照。那两个网络**完全不知道 ESM 的存在**：每个残基只告诉网络"我是哪种氨基酸"和 6 个结构特征，突变只告诉它替换本身的 5 个特征。

为什么要有第二行：我猜图网络的用处是**给残基补上周围的信息**。ESM 的向量可能已经带着周围的信息了，那样的话在它上面再看邻居就是多此一举。要验证这个猜想，就得有一组"什么周围信息都没有"的输入来对比。

### 用了什么技术，为什么
**这一章不用 Mac 的 GPU，用 CPU。这是一次真实的事故换来的。**

第一次我照 08a 的样子把图网络放到 Mac 的 GPU 上训练，跑了十几分钟，这个进程占了 **30 GB 内存**，把你 Mac 的交换区撑满，整台机器都慢了下来。我把它杀掉以后查原因：

- PyTorch 在 Apple GPU 上，会给**每一种输入大小**编译并缓存一份计算方案，而且不释放
- 08a 每批固定 4,096 个突变，输入大小永远一样，所以只缓存一份，没事
- 图网络每一批是 8 个蛋白，**每一批的节点数、边数都不一样**，于是每一步都多缓存一份，越攒越多

解决办法很简单：这个网络小，图又裁过，**在 CPU 上每轮只要 1 秒，和 GPU 一样快**，内存稳定在 2.5 GB。所以 `graph.device()` 在 Mac 上直接返回 CPU（有 NVIDIA 显卡的机器不受影响）。

### 执行
四个网络，同一张卷子（和 07f、08a 完全相同的 5 折）。约 13 分钟：

In [ ]:
REFERENCES = {name: tracking.read_scores(w, run) for name, run in
              [("zero_shot", "esm2_zero_shot"), ("baseline", "baseline"), ("mlp_standardised", "mlp_standardised")]}
predictions, scores = {}, {}
for name, config in graph.RUNS.items():
    predictions[name] = graph.cross_validate(train, graphs, **config)
    scores[name] = baseline.dataset_scores(predictions[name])
    twin = name.replace("2_layers", "0_layers")                             # 同样的输入、不看邻居的那一个
    versus = {"no_graph_layers": scores[twin]} if twin != name else {}      # 看邻居的，和它的"双胞胎"配对比较
    tracking.cv_run(w, EXPERIMENT, name, {**graph.parameters(config), "feature_table_version": VERSION, "esm2_revision": ESM2},
                    scores[name], references={**REFERENCES, **versus}, chapter="08b")
    print(name, baseline.summarise(scores[name]), "\n")
print("存回平台的预测:", f"{tracking.save_predictions(w, predictions['gnn_structure_2_layers'], 'gnn_structure_2_layers'):,}", "行")

### 结果怎么看
四个运行各打印一行成绩和一个 MLflow 链接。下一步把它们排成表来看。

### 公司里什么情况怎么做
- **新硬件、新后端要盯着内存看**。这次的问题在 NVIDIA 显卡上不存在，在 Apple GPU 上才有。"在我机器上能跑"和"能一直跑"是两回事
- **把事故写下来**。原因、现象、怎么查的、怎么修的。下一个人（包括三个月后的自己）遇到"Mac 上训练越来越慢"，能直接找到答案
- 设计对照时问自己：**如果两个结果不一样，我能说出是哪一个改动造成的吗？** 不能的话，实验就没设计好

### 面试怎么说
> "To isolate the effect of message passing I ran a 2 by 2 design: with and without graph layers, on ESM embeddings and on structure-only inputs, same folds and scoring. I also hit an Apple-GPU issue: variable-size graph batches made the MPS backend cache a compiled graph per shape until memory reached 30 GB, so graph training runs on CPU, which is just as fast for this model."

---

## 第 5 步 · 看结果

### 在干什么
把 2 × 2 的四个数填进表里。

### 执行

In [ ]:
runs = tracking.latest_runs(EXPERIMENT)
runs.loc[[*graph.RUNS, "mlp_standardised"], ["all", "domainome", "proteingym", "mavedb"]].round(3)

### 结果怎么看
每组实验的 Spearman 的中位数：

| 运行 | 全部 | domainome | proteingym | mavedb（功能） |
|---|---|---|---|---|
| `gnn_esm_0_layers` | 0.626 | 0.654 | 0.664 | 0.387 |
| `gnn_esm_2_layers` | 0.615 | 0.647 | 0.616 | 0.361 |
| `gnn_structure_0_layers` | 0.529 | 0.553 | 0.548 | 0.340 |
| `gnn_structure_2_layers` | 0.558 | 0.581 | 0.547 | 0.347 |
| `mlp_standardised`（08a） | 0.627 | 0.652 | 0.634 | 0.378 |

先核对一件事：`gnn_esm_0_layers`（0.626）和 08a 的 `mlp_standardised`（0.627）几乎一样。应该如此，它们本来就是同一种网络，只是这一章按蛋白分批训练。这说明新代码没有引入别的差别。

把"全部"那一列填回 2 × 2 的表：

| | 不看邻居 | 看邻居 | 看邻居带来的变化 |
|---|---|---|---|
| **有 ESM 向量** | 0.626 | 0.615 | **变差了** |
| **没有 ESM，只有结构** | 0.529 | 0.558 | **变好了** |

同一个改动，在两种输入上效果相反。配对比较确认一下，先看有 ESM 的，再看没有 ESM 的：

In [ ]:
display(tracking.comparison(runs, "gnn_esm_2_layers", "no_graph_layers"))
tracking.comparison(runs, "gnn_structure_2_layers", "no_graph_layers")

### 结果怎么看
**有 ESM 向量时，看邻居帮倒忙**（第一张表）：

| 数据 | 看邻居更好的实验占 | 平均变化 | p 值 |
|---|---|---|---|
| domainome | 35% | −0.008 | < 0.001 |
| proteingym | 0%（10 组全输） | −0.038 | 0.002 |
| mavedb（功能） | 8% | −0.033 | < 0.001 |

**没有 ESM 时，看邻居有用**（第二张表）：

| 数据 | 看邻居更好的实验占 | 平均变化 | p 值 |
|---|---|---|---|
| domainome | 81% | +0.032 | < 0.001 |
| proteingym | 60% | +0.016 | 0.43，不显著 |
| mavedb（功能） | 56% | +0.007 | 0.14，不显著 |

三个结论：

1. **图网络本身是管用的。** 输入里没有周围的信息时，让残基看邻居，稳定性预测明显变好（81% 的 Domainome 实验更好）
2. **但在 ESM 向量上面再加图层，反而变差。** 图层多出来二十几万个要训练的参数，却没带来新信息，于是只是多了过拟合的机会。我们只有 279 个训练蛋白，经不起
3. **顺带一个发现**：完全不用语言模型、只看邻居的图网络有 0.558，比 07f 那个用了 ESM 打分的树模型基线（0.550）还高一点。结构本身的信息量不小

猜想对了一半还不够，下一步直接去查：ESM 的向量里，到底是不是已经有了周围的信息。

### 公司里什么情况怎么做
- **更复杂的模型不一定更好**。加一个组件之前先问：它带来了什么**新信息**？如果答不上来，它带来的多半只是更多参数和更多过拟合
- **负结果要认真报告**。"我们试了图网络，没用，原因是……"能帮团队省下几周。很多团队反复掉进同一个坑，就是因为上一个人没写下来

### 面试怎么说
> "Graph layers improved a structure-only model, plus 0.03 Spearman on stability assays, but made the ESM-based model slightly worse on all three assay types. With 279 training proteins, extra parameters without new information only add variance."

---

## 第 6 步 · 为什么：探针实验

### 在干什么
直接检查 ESM 的向量里有没有三维结构的信息。

### 背景
比喻：**查一查专家的笔记里已经写了什么。** ESM-2 只读过序列，从没见过一个三维结构。那它给每个残基写的 1,280 个数里，会不会已经含有"这个残基埋不埋、周围挤不挤"？

检查的办法叫**探针**（probe），是研究大模型"到底学到了什么"的标准手段：
- 用**最简单**的模型（一条直线，也就是给 1,280 个数各配一个权重再相加），试着从向量里读出某个性质，比如 `rsa`
- 在一部分蛋白上定权重，在**没见过的蛋白**上看读得准不准
- 读得准，说明这个信息就**明摆在**向量里；要很复杂的模型才读得出来的不算

为什么非得用最简单的模型：模型一复杂，就分不清是"向量里本来有"，还是"模型自己算出来的"。

### 执行

In [ ]:
groups = train.groupby("uniprot_id")["similarity_group"].first()
columns = {"rsa": "埋不埋", "contacts_12": "12 Å 内有多少残基", "plddt": "AlphaFold 有多确定", "hse_up": "侧链那一侧有多挤"}
pd.DataFrame({"性质": columns, "直线读出来的值和真实值的相关系数": {c: graph.probe(table, index, residues, c, groups) for c in columns}})

### 结果怎么看
在没见过的蛋白上，一条直线从 ESM 向量里读出来的值，和 AlphaFold 结构里量出来的真实值的相关系数：

| 性质 | 相关系数 |
|---|---|
| `rsa`，埋不埋 | 0.87 |
| `contacts_12`，周围有多少残基 | 0.90 |
| `plddt`，AlphaFold 有多确定 | 0.91 |
| `hse_up`，侧链那一侧有多挤 | 0.86 |

**全在 0.86 以上。** ESM-2 从没见过结构，但它的向量里已经把"埋不埋""挤不挤"写得清清楚楚，一条直线就读得出来。

这一下三件事都说得通了：
- **08a**：给向量再拼上 16 个特征（其中 6 个是结构特征），只涨了 0.01。因为向量里已经有了
- **这一章**：在向量上再加图层没用。因为图层要补的"周围的信息"，向量里已经有了
- **07f**："周围环境"那组特征没证明自己有用。同理

**为什么 ESM 会知道结构**：ESM-2 是 Transformer，它的核心部件叫**自注意力**（self-attention）：每一层里，每个残基都去"看"序列上**所有**其他残基，并且自己决定重点听谁的。这其实就是消息传递，只不过图是"所有残基两两相连"的，而且它做了 **33 层**。在几千万条序列上训练以后，它学会了哪些残基总是一起变（共进化），而一起变的残基往往在空间里挨着。**ESM-2 自己早就做过消息传递了，做得比我们的两层图层彻底得多。**

### 公司里什么情况怎么做
- **用预训练大模型之前，先探一探它已经知道什么。** 探针实验几十秒就能跑完，能避免花几周去造一个大模型早就会的东西
- 这也是为什么这几年很多领域里，"大模型的向量 + 一个简单的头"打败了精心设计的专用结构

### 面试怎么说
> "A linear probe on frozen ESM-2 embeddings recovers solvent accessibility, contact density and pLDDT of held-out proteins with correlations of 0.86 to 0.91, although ESM-2 never saw a structure. Self-attention is message passing on a fully connected graph, 33 layers deep, so a structural GNN on top adds nothing new."

---

## 第 7 步 · 图网络能帮上集成吗

### 在干什么
最后给图网络一次机会：它单独用不如 08a 的网络，但加进集成会不会有帮助？

### 背景
08a 第 7 步说过，集成有用的前提是**成员犯的错不一样**。`gnn_structure_2_layers` 是我们手里唯一一个**完全没碰过语言模型**的模型，它的信息来源和其他三个都不同，按道理是个好的会诊对象。试一试。

### 执行
取出前两章存在平台上的预测，和它排在一起：

In [ ]:
lakehouse.download(w, f"SELECT * FROM {CATALOG}.ml.cv_predictions WHERE run_name IN ('baseline', 'mlp_standardised')", MODELS / "earlier_predictions.parquet")
earlier = pd.read_parquet(MODELS / "earlier_predictions.parquet").pivot(index=baseline.KEY, columns="run_name", values="predicted").reset_index()
members = predictions["gnn_structure_2_layers"].rename(columns={"predicted": "gnn_structure"}).merge(earlier, on=baseline.KEY)
combos = {"08a 的集成": ["esm_delta", "baseline", "mlp_standardised"], "再加上结构图网络": ["esm_delta", "baseline", "mlp_standardised", "gnn_structure"]}
combined = {name: baseline.dataset_scores(members.assign(predicted=baseline.rank_average(members, columns))) for name, columns in combos.items()}
short = lambda column: column.replace("spearman_median_", "").replace("spearman_median", "all")
display(pd.DataFrame({name: baseline.summarise(s) for name, s in combined.items()}).T.drop(columns="datasets").rename(columns=short).round(3))
pd.Series(baseline.compare(combined["再加上结构图网络"], combined["08a 的集成"])).filter(like="mavedb")

### 结果怎么看
| | 全部 | domainome | proteingym | mavedb（功能） |
|---|---|---|---|---|
| 08a 的集成 | 0.592 | 0.617 | 0.613 | 0.432 |
| 再加上结构图网络 | 0.611 | 0.633 | 0.624 | 0.437 |

- 我们用集成是为了**功能**这一栏。中位数从 0.432 到 0.437，但一组一组配对比较，平均只高了不到 0.001，p = 0.25：**没有差别**
- 稳定性那两栏高了一点，但稳定性我们本来就不用集成，用的是 08a 的网络（0.652），比这里的 0.633 高

所以这位新大夫没有提出新意见。**最终用什么模型，这一章之后没有变：**

| 想问什么 | 用哪个 | 成绩 |
|---|---|---|
| 会不会不稳定 | `mlp_standardised`（08a） | 0.65 |
| 会不会失去功能 | `ensemble`（08a） | 0.43 |

**这一章的价值不在成绩上**，在于现在能有根据地说：
- 图神经网络是什么、什么时候有用（输入缺周围信息时）
- 为什么它在我们这里没用（ESM 的向量已经有了，探针实验为证）
- 所以想再往上走，不该在"加什么结构"上花力气，而要么**改进向量本身**（08c 微调），要么**找更多功能数据**

### 公司里什么情况怎么做
- 做完一个没带来提升的方向，**要有一个明确的"不采用"的决定和理由**，而不是让代码不清不楚地留在那里。我们的决定：图网络的代码和测试留着（以后换成没有好向量的数据时用得上），但不进最终模型
- 评估一个新成员要看它能不能改进**你真正在用的那个指标**，不是随便哪个指标

### 面试怎么说
> "I also tested the structure-only GNN as an ensemble member because it was the only model without language-model input. It did not move the functional score, 0.437 versus 0.432, p of 0.25, so the production choice stayed the same and the next lever is the representation itself or more functional data."

---

## 提交代码

In [ ]:
run("git add -A")
run("uv run pre-commit run", check=False)   # 第一次会自动整理 notebook 的格式
run("git add -A")
run('(git diff --cached --quiet || git commit -q -m "feat: 08b graph neural network on AlphaFold residue graphs") && git push -q')

推送后去 GitHub 的 Actions 页面看一眼，应该全绿。

---

## 本章小结

| 步骤 | 做了什么 | 记住什么 |
|---|---|---|
| 1 图 | 残基是节点，10 Å 以内连一条边 | 序列上远、空间上近的残基被连起来了 |
| 2 消息传递 | 每个残基合并邻居的平均 | 几层就传几步 |
| 3 建图 | PyG 的 `Data`，裁到两步以内 | 只留 17% 的节点，结果不变，有测试 |
| 4 对照 | 2 × 2：有没有 ESM × 看不看邻居 | 一次只改一样；Apple GPU 的内存事故 |
| 5 结果 | 没有 ESM 时有用，有 ESM 时帮倒忙 | 多的参数没带来新信息，就只带来过拟合 |
| 6 探针 | 一条直线就能从向量里读出结构，0.86–0.91 | 自注意力就是消息传递，ESM 早做过了 |
| 7 集成 | 加进去功能不变 | 最终模型没变，方向清楚了 |

**一句话**：图网络能给"没有周围信息"的输入补上周围信息；ESM 的向量已经有了，所以在它上面加图网络没用。

## 面试题

1. **What is message passing?**
   > "Each node updates its vector from its own and an aggregate of its neighbours', through learned weights. k layers propagate information k hops."

2. **Your GNN did not beat the MLP. What did you conclude?**
   > "I ran a 2 by 2 ablation and a probe. Graph layers helped a structure-only model but hurt the ESM-based one, and a linear probe showed ESM embeddings already encode burial and contact density at 0.9 correlation. So the graph added parameters, not information."

3. **What is a linear probe and why keep it linear?**
   > "A simple model trained to read a property out of frozen embeddings. Keeping it linear shows the information is explicitly present; a powerful probe could compute it by itself."

4. **How is a Transformer related to a GNN?**
   > "Self-attention is message passing on a fully connected graph with learned edge weights. That is why a protein language model can pick up structural contacts from sequences alone."

5. **Tell me about a performance or memory problem you debugged.**
   > "Training a GNN on Apple's GPU grew to 30 GB because the MPS backend caches a compiled graph for every new input shape, and graph batches never repeat a shape. I measured it fold by fold, saw GPU memory flat and process memory climbing, and moved the model to CPU, which was equally fast after pruning the graphs to the two-hop neighbourhood of labelled residues."

**下一步 08c · 微调 ESM-2**：这一章的结论是"要改进，就得改进向量本身"。08a 和 08b 里 ESM-2 一直是冻结的。08c 把它解冻一小部分（LoRA），让我们的标签去改它自己。